In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print(OPENAI_API_KEY[:2])

UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
print(UPSTAGE_API_KEY[30:])

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
print(TAVILY_API_KEY[:4])

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import re
import warnings
from pprint import pprint
from typing import List
from textwrap import dedent

from langchain_community.document_loaders import TextLoader, WikipediaLoader
from langchain_community.tools import TavilySearchResults
from langchain_community.vectorstores import FAISS
from langchain_upstage import UpstageEmbeddings
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableConfig, chain
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_upstage import ChatUpstage
from pydantic import BaseModel, Field

In [ ]:
# 1. 카페 메뉴 데이터 파일 생성 및 벡터 DB 구축
def create_cafe_vector_db():
    os.makedirs("../data", exist_ok=True)
    os.makedirs("../db", exist_ok=True)

    cafe_menu_content = """
    1. 아메리카노
    • 가격: ₩4,500
    • 주요 원료: 에스프레소, 뜨거운 물
    • 설명: 진한 에스프레소에 뜨거운 물을 더해 만든 클래식한 블랙 커피입니다. 원두 본연의 맛을 가장 잘 느낄 수 있으며, 깔끔하고 깊은 풍미가 특징입니다. 설탕이나 시럽 추가 가능합니다.

    2. 카페라떼
    • 가격: ₩5,500
    • 주요 원료: 에스프레소, 스팀 밀크
    • 설명: 진한 에스프레소에 부드럽게 스팀한 우유를 넣어 만든 대표적인 밀크 커피입니다. 크리미한 질감과 부드러운 맛이 특징이며, 다양한 시럽과 토핑 추가가 가능합니다. 라떼 아트로 시각적 즐거움도 제공합니다.

    3. 카푸치노
    • 가격: ₩5,000
    • 주요 원료: 에스프레소, 스팀 밀크, 우유 거품
    • 설명: 에스프레소, 스팀 밀크, 우유 거품이 1:1:1 비율로 구성된 이탈리아 전통 커피입니다. 진한 커피 맛과 부드러운 우유 거품의 조화가 일품이며, 계피 파우더를 뿌려 제공합니다.

    4. 바닐라 라떼
    • 가격: ₩6,000
    • 주요 원료: 에스프레소, 스팀 밀크, 바닐라 시럽
    • 설명: 카페라떼에 달콤한 바닐라 시럽을 더한 인기 메뉴입니다. 바닐라의 달콤함과 커피의 쌉싸름함이 조화롭게 어우러지며, 휘핑크림 토핑으로 더욱 풍성한 맛을 즐길 수 있습니다.

    5. 카라멜 마키아토
    • 가격: ₩6,500
    • 주요 원료: 에스프레소, 스팀 밀크, 카라멜 시럽, 휘핑크림
    • 설명: 스팀 밀크 위에 에스프레소를 부어 만든 후 카라멜 시럽과 휘핑크림으로 마무리한 달콤한 커피입니다. 카라멜의 진한 단맛과 커피의 깊은 맛이 조화를 이루며, 시각적으로도 아름다운 층을 형성합니다.

    6. 콜드브루
    • 가격: ₩5,000
    • 주요 원료: 콜드브루 원액, 차가운 물
    • 설명: 찬물에 12-24시간 우려낸 콜드브루 원액을 사용한 시원한 커피입니다. 부드럽고 달콤한 맛이 특징이며, 산미가 적어 누구나 부담 없이 즐길 수 있습니다. 얼음과 함께 시원하게 제공됩니다.

    7. 프라푸치노
    • 가격: ₩7,000
    • 주요 원료: 에스프레소, 우유, 얼음, 휘핑크림
    • 설명: 에스프레소와 우유, 얼음을 블렌더에 갈아 만든 시원한 음료입니다. 부드럽고 크리미한 질감이 특징이며, 휘핑크림을 올려 달콤함을 더했습니다. 여름철 인기 메뉴입니다.

    8. 녹차 라떼
    • 가격: ₩5,800
    • 주요 원료: 말차 파우더, 스팀 밀크, 설탕
    • 설명: 고급 말차 파우더와 부드러운 스팀 밀크로 만든 건강한 음료입니다. 녹차의 은은한 쓴맛과 우유의 부드러움이 조화를 이루며, 항산화 성분이 풍부합니다. 달콤함 조절이 가능합니다.

    9. 아이스 아메리카노
    • 가격: ₩4,500
    • 주요 원료: 에스프레소, 차가운 물, 얼음
    • 설명: 진한 에스프레소에 차가운 물과 얼음을 넣어 만든 시원한 아이스 커피입니다. 깔끔하고 시원한 맛이 특징이며, 원두 본연의 풍미를 느낄 수 있습니다. 더운 날씨에 인기가 높습니다.

    10. 티라미수
        • 가격: ₩7,500
        • 주요 원료: 마스카포네 치즈, 에스프레소, 레이디핑거, 코코아 파우더
        • 설명: 이탈리아 전통 디저트로 마스카포네 치즈와 에스프레소에 적신 레이디핑거를 층층이 쌓아 만들었습니다. 부드럽고 달콤한 맛이 특징이며, 코코아 파우더로 마무리하여 깊은 풍미를 자랑합니다.
    """

    with open("../data/cafe_menu.txt", "w", encoding="utf-8") as f:
        f.write(cafe_menu_content)

    loader = TextLoader("../data/cafe_menu.txt", encoding="utf-8")
    documents = loader.load()

    all_menu_docs = []
    for doc in documents:
        all_menu_docs.extend(split_menu_items(doc))

    embeddings_model = UpstageEmbeddings(model="solar-embedding-1-large")

    print("벡터 DB를 생성하고 있습니다...")
    menu_db = FAISS.from_documents(
        documents=all_menu_docs,
        embedding=embeddings_model
    )
    menu_db.save_local("../db/cafe_db")
    print("벡터 DB 생성이 완료되었습니다: '../db/cafe_db'")

def split_menu_items(document: Document) -> List[Document]:
    content = document.page_content
    pattern = r'(\d+\.\s.*?)(?=\n\n\d+\.|$)'
    items = re.findall(pattern, document.page_content, re.DOTALL)

    menu_documents = []
    for i, item in enumerate(items, 1):
        menu_name = item.split('\n')[0].split('.', 1)[1].strip()
        menu_doc = Document(
            page_content=item.strip(),
            metadata={
                "source": document.metadata['source'],
                "menu_number": i,
                "menu_name": menu_name
            }
        )
        menu_documents.append(menu_doc)
    return menu_documents

In [ ]:
@tool
def tavily_search_func(query: str) -> str:
    tavily_search = TavilySearchResults(max_results=3)
    docs = tavily_search.invoke(query)
    formatted_docs = "\n---\n".join([
        f'<Document href="{doc["url"]}">\n{doc["content"]}\n</Document>'
        for doc in docs
    ])
    if len(formatted_docs) > 0:
        return formatted_docs
    
    return "관련 정보를 찾을 수 없습니다."

def wiki_search_and_summarize(input_data: dict) -> List[str]:
    wiki_loader = WikipediaLoader(query=input_data["query"], load_max_docs=1, lang="ko")
    wiki_docs = wiki_loader.load()
    return [
        f'<Document source="{doc.metadata["source"]}">\n{doc.page_content}\n</Document>'
        for doc in wiki_docs
    ]

@tool
def wiki_summary(query: str) -> str:
    summary_prompt = ChatPromptTemplate.from_template(
        "Summarize the following text in a concise manner:\n\n{context}\n\nSummary:"
    )

    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0
    )

    summary_chain = (
        {"context": RunnableLambda(wiki_search_and_summarize)}
        | summary_prompt
        | llm
        | StrOutputParser()
    )
    
    summary = summary_chain.invoke({"query":query})
    
    return summary

@tool
def db_search_cafe_func(query: str) -> List[Document]:
    try:
        embeddings_model = UpstageEmbeddings(model="solar-embedding-1-large")
        db = FAISS.load_local(
            "./db/cafe_db",
            embeddings_model,
            allow_dangerous_deserialization=True
        )
        docs = db.similarity_search(query, k=3)
        return docs if docs else [Document(page_content="관련 메뉴 정보를 찾을 수 없습니다.")]
    except Exception as e:
        return [Document(page_content=f"DB 검색 중 오류가 발생했습니다: {e}")]

llm = ChatUpstage(
        model="solar-pro",
        base_url="https://api.upstage.ai/v1",
        temperature=0.5
)

tools = [tavily_search_func, wiki_summary, db_search_cafe_func]
llm_with_tools = llm.bind_tools(tools=tools)

In [ ]:

prompt = ChatPromptTemplate([
    ("system", "당신은 카페 메뉴 정보를 제공하는 유용한 AI 어시스턴트입니다."),
    ("human", "{user_input}"),
    ("placeholder", "{messages}"),
])

llm_chain = prompt | llm_with_tools

@chain
def cafe_assistant_chain(user_input: str, config: RunnableConfig):
    input_ = {"user_input": user_input}
    
    ai_msg = llm_chain.invoke(input_, config=config)
    
    if not ai_msg.tool_calls:
        return ai_msg

    tool_msgs = []
    for tool_call in ai_msg.tool_calls:
        print(f"호출된 도구: {tool_call['name']}({tool_call['args']})")
        
        if tool_call["name"] == "tavily_search_func":
            tool_message = tavily_search_func.invoke(tool_call, config=config)
        elif tool_call["name"] == "wiki_summary":
            tool_message = wiki_summary.invoke(tool_call, config=config)
        elif tool_call["name"] == "db_search_cafe_func":
            tool_message = db_search_cafe_func.invoke(tool_call, config=config)
        else:
            tool_message = f"알 수 없는 도구: {tool_call['name']}"

        tool_msgs.append(tool_message)

    return llm_chain.invoke({**input_, "messages": [ai_msg, *tool_msgs]}, config=config)

In [ ]:
if __name__ == "__main__":
    if not os.path.exists("../db/cafe_db"):
        create_cafe_vector_db()

    print("\n--- 도구 정의 확인 ---")
    for tool_item in tools:
        print(f"이름: {tool_item.name}")
        print(f"설명: {tool_item.description.strip()}")
        print("-" * 20)

    print("\n--- 테스트 시작 ---")
    query = "아메리카노의 가격과 특징은 무엇인가요?"
    print(f"질문: {query}\n")

    response = cafe_assistant_chain.invoke(query)

    print("\n--- 최종 답변 ---")
    pprint(response.content)

    print("\n\n--- 추가 테스트 ---")
    query_2 = "커피의 역사에 대해 알려주고, 요즘 유행하는 커피 트렌드도 알려줘."
    print(f"질문: {query_2}\n")

    response_2 = cafe_assistant_chain.invoke(query_2)

    print("\n--- 최종 답변 ---")
    pprint(response_2.content)